# Deploy UCRG Agent to Databricks Model Serving

Packages the Use-Case Requirement Gathering agent as an MLflow **pyfunc** model and deploys it to a **Model Serving endpoint**.

**Protocol (stateless, multi-replica safe)**
1. `action=start` → greeting + first question + `session_state`
2. `action=send` + user `message` + prior `session_state` → next turn
3. When `done=true`, parse `output_json` for SDD + scorecard markdown

Run locally in VS Code / Jupyter to smoke-test. Databricks-only cells auto-skip outside Databricks.

In [ ]:
# --- configuration -------------------------------------------------------
from pathlib import Path

IN_DATABRICKS = False
try:
    dbutils  # type: ignore[name-defined]
    IN_DATABRICKS = True
except NameError:
    pass

if IN_DATABRICKS:
    dbutils.widgets.text("catalog", "main", "Unity Catalog")
    dbutils.widgets.text("schema", "ucrg", "Schema")
    dbutils.widgets.text("model_name", "ucrg_agent", "Registered model name")
    dbutils.widgets.text("endpoint_name", "ucrg-agent", "Serving endpoint name")
    dbutils.widgets.dropdown("llm_backend", "mock", ["anthropic", "mock"], "LLM backend")
    dbutils.widgets.text("secret_scope", "ucrg", "Secret scope")
    dbutils.widgets.text("secret_key", "anthropic_api_key", "Secret key")
    dbutils.widgets.text("project_root", "", "Repo root (blank = auto-detect)")

    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA = dbutils.widgets.get("schema")
    MODEL_NAME = dbutils.widgets.get("model_name")
    ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
    LLM_BACKEND = dbutils.widgets.get("llm_backend")
    SECRET_SCOPE = dbutils.widgets.get("secret_scope")
    SECRET_KEY = dbutils.widgets.get("secret_key")
else:
    CATALOG = "main"
    SCHEMA = "ucrg"
    MODEL_NAME = "ucrg_agent"
    ENDPOINT_NAME = "ucrg-agent"
    LLM_BACKEND = "mock"  # use "anthropic" locally if ANTHROPIC_API_KEY is set
    SECRET_SCOPE = "ucrg"
    SECRET_KEY = "anthropic_api_key"
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "ucrg").exists() and (PROJECT_ROOT.parent / "ucrg").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

FULL_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
print(f"Runtime: {'Databricks' if IN_DATABRICKS else 'local'}")
if not IN_DATABRICKS:
    print(f"Project root: {PROJECT_ROOT}")
print(f"Model: {FULL_MODEL_NAME}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Backend: {LLM_BACKEND}")
if IN_DATABRICKS:
    print("Project root: auto-detected after pip install (next cells)")

In [ ]:
# --- install dependencies ------------------------------------------------
# Only install packages missing from the cluster runtime. Upgrading pandas/mlflow
# here can break pre-installed notebook deps (protobuf, numpy, etc.).
%pip install -q anthropic>=0.39 requests>=2.31

if IN_DATABRICKS:
    dbutils.library.restartPython()

In [ ]:
# --- reload config + imports (required after restartPython on Databricks) ---
import json
import os
import sys
from pathlib import Path

import pandas as pd

IN_DATABRICKS = False
try:
    dbutils  # type: ignore[name-defined]
    IN_DATABRICKS = True
except NameError:
    pass


def _resolve_project_root(widget_value: str) -> Path:
    cleaned = (widget_value or "").strip()
    if cleaned:
        root = Path(cleaned)
        if (root / "ucrg").is_dir():
            return root
        raise FileNotFoundError(
            f"project_root widget is {root} but ucrg/ was not found. "
            "Clear the widget to auto-detect."
        )

    nb_path = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )
    if not nb_path.startswith("/Workspace"):
        nb_path = f"/Workspace{nb_path}"

    cursor = Path(nb_path).parent
    for _ in range(6):
        if (cursor / "ucrg").is_dir() and (cursor / "data" / "ucrg_engine.json").is_file():
            return cursor
        if cursor.parent == cursor:
            break
        cursor = cursor.parent

    raise FileNotFoundError(
        f"Could not auto-detect repo root from {nb_path!r}. "
        "Set project_root to e.g. /Workspace/Repos/<user>/cusstom_uc_agent"
    )


if IN_DATABRICKS:
    CATALOG = dbutils.widgets.get("catalog")
    SCHEMA = dbutils.widgets.get("schema")
    MODEL_NAME = dbutils.widgets.get("model_name")
    ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
    LLM_BACKEND = dbutils.widgets.get("llm_backend")
    SECRET_SCOPE = dbutils.widgets.get("secret_scope")
    SECRET_KEY = dbutils.widgets.get("secret_key")
    PROJECT_ROOT = _resolve_project_root(dbutils.widgets.get("project_root"))
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "ucrg").exists() and (PROJECT_ROOT.parent / "ucrg").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

FULL_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from ucrg.serving import UCRGServingModel


class ModelContext:
    artifacts = {
        "ucrg_engine": str(PROJECT_ROOT / "data" / "ucrg_engine.json"),
        "system_prompt": str(PROJECT_ROOT / "prompts" / "system_prompt.md"),
    }
    model_config = {"backend": LLM_BACKEND}


def invoke_model(model: UCRGServingModel, ctx: ModelContext, record: dict) -> dict:
    frame = model.predict(ctx, pd.DataFrame([record]))
    return frame.iloc[0].to_dict()


model = UCRGServingModel()
model.load_context(ModelContext())
print("Serving model loaded.")

In [ ]:
# --- smoke test ----------------------------------------------------------
start = invoke_model(model, ModelContext(), {
    "action": "start",
    "session_id": "smoke-test",
    "message": "",
    "session_state": "",
})

assert start["session_state"], "expected session_state blob"
print(start["message"][:240], "…")
print("Smoke test OK")

In [ ]:
# --- full demo interview (mock backend, runs locally) --------------------
DEMO_ANSWERS = [
    "I want a tool that analyzes sales data and predicts next quarter revenue trends.",
    "We handle forecasting manually in spreadsheets today.",
    "Jane Smith, Head of Sales Analytics",
    "About 15 sales managers, weekly",
    "No, it only affects internal planning reports",
    "No fairness concerns",
    "It should read our CRM export and produce a forecast dashboard",
    "It suggests actions but a human approves before anything changes",
    "Mostly rules-based calculations with some trend analysis",
    "Single step - upload file, get report",
    "It analyzes data and predicts outcomes",
    "We have historical sales CSV files from Salesforce",
    "Yes, customer names and deal amounts are included",
    "No, it only reads the uploaded file",
    "It connects to Salesforce read-only",
    "No external sharing",
    "A human reviews every forecast before sharing",
    "We follow GDPR for customer data",
    "Yes, GDPR applies to customer personal data in deals",
    "We need audit logs of who ran forecasts",
    "Success means forecasts within 10% of actuals",
    "We want to scale to 50 users next year",
    "Low impact if wrong - just bad planning estimates",
    "Sales revenue data owned by Finance",
    "No purchased datasets",
    "Historical deal data from CRM",
    "Salesforce CSV exports, good quality",
    "Yes personal customer data included",
    "No internal knowledge base needed",
    "Salesforce read only",
    "No sharing outside the team",
]

turn = invoke_model(model, ModelContext(), {
    "action": "start",
    "session_id": "demo",
    "message": "",
    "session_state": "",
})
session_state = turn["session_state"]
print("Agent:", turn["message"][:160], "…\n")

for answer in DEMO_ANSWERS:
    turn = invoke_model(model, ModelContext(), {
        "action": "send",
        "session_id": "demo",
        "message": answer,
        "session_state": session_state,
    })
    session_state = turn["session_state"]
    if turn.get("error"):
        raise RuntimeError(turn["error"])
    if turn["done"]:
        print("Agent:", turn["message"])
        break

assert turn["done"], "demo interview did not complete"
output = json.loads(turn["output_json"])

out_dir = PROJECT_ROOT / "output"
out_dir.mkdir(exist_ok=True)
(out_dir / "notebook_demo_SDD.md").write_text(output["sdd"], encoding="utf-8")
(out_dir / "notebook_demo_scorecard.md").write_text(output["scorecard"], encoding="utf-8")
print("\nWritten:")
print("  output/notebook_demo_SDD.md")
print("  output/notebook_demo_scorecard.md")

In [ ]:
# --- log & register model (Databricks UC) --------------------------------
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

if not IN_DATABRICKS:
    print("Skipping UC registration — not running on Databricks.")
else:
    mlflow.set_registry_uri("databricks-uc")

    input_example = pd.DataFrame([{
        "action": "start",
        "session_id": "example",
        "message": "",
        "session_state": "",
    }])
    output_example = pd.DataFrame([{
        "session_id": "example",
        "message": "Hi! …",
        "done": False,
        "session_state": "{}",
        "output_json": "",
        "error": "",
    }])
    signature = infer_signature(input_example, output_example)

    with mlflow.start_run(run_name="ucrg-serving") as run:
        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=UCRGServingModel(),
            code_paths=[
                str(PROJECT_ROOT / "ucrg"),
                str(PROJECT_ROOT / "data"),
                str(PROJECT_ROOT / "prompts"),
            ],
            artifacts={
                "ucrg_engine": str(PROJECT_ROOT / "data" / "ucrg_engine.json"),
                "system_prompt": str(PROJECT_ROOT / "prompts" / "system_prompt.md"),
            },
            pip_requirements=["anthropic>=0.39", "pandas>=2.0"],
            signature=signature,
            input_example=input_example,
            model_config={"backend": LLM_BACKEND},
        )
        run_id = run.info.run_id

    model_uri = f"runs:/{run_id}/model"
    registered = mlflow.register_model(model_uri, FULL_MODEL_NAME)
    version = registered.version
    print(f"Registered {FULL_MODEL_NAME} version {version}")

    client = MlflowClient()
    client.set_registered_model_alias(FULL_MODEL_NAME, "champion", version)

In [ ]:
# --- create or update serving endpoint -----------------------------------
if not IN_DATABRICKS:
    print("Skipping endpoint deployment — not running on Databricks.")
else:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

    w = WorkspaceClient()

    served_entity = ServedEntityInput(
        entity_name=FULL_MODEL_NAME,
        entity_version=version,
        workload_size="Small",
        scale_to_zero_enabled=True,
        environment_vars={
            "UCRG_LLM_BACKEND": LLM_BACKEND,
            "ANTHROPIC_API_KEY": f"{{{{secrets/{SECRET_SCOPE}/{SECRET_KEY}}}}}",
        },
    )
    endpoint_config = EndpointCoreConfigInput(served_entities=[served_entity])

    existing = [ep for ep in w.serving_endpoints.list() if ep.name == ENDPOINT_NAME]
    if existing:
        print(f"Updating endpoint {ENDPOINT_NAME} → v{version}")
        w.serving_endpoints.update_config_and_wait(
            name=ENDPOINT_NAME,
            served_entities=endpoint_config.served_entities,
        )
    else:
        print(f"Creating endpoint {ENDPOINT_NAME}")
        w.serving_endpoints.create_and_wait(name=ENDPOINT_NAME, config=endpoint_config)

    print("Endpoint ready:", ENDPOINT_NAME)

In [ ]:
# --- invoke deployed endpoint (REST) -------------------------------------
if not IN_DATABRICKS:
    print("Skipping REST invoke — not running on Databricks.")
else:
    import requests

    host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    url = f"{host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    def invoke_endpoint(records: list[dict]) -> list[dict]:
        payload = {"dataframe_records": records}
        resp = requests.post(url, headers=headers, json=payload, timeout=120)
        resp.raise_for_status()
        body = resp.json()
        preds = body.get("predictions", body)
        if isinstance(preds, dict) and "data" in preds:
            cols = preds["columns"]
            return [dict(zip(cols, row)) for row in preds["data"]]
        return preds

    turn = invoke_endpoint([{
        "action": "start",
        "session_id": "endpoint-demo",
        "message": "",
        "session_state": "",
    }])[0]
    print("Agent:", turn["message"][:300], "…")

    turn = invoke_endpoint([{
        "action": "send",
        "session_id": "endpoint-demo",
        "message": "I want a chatbot that answers HR policy questions from our internal wiki.",
        "session_state": turn["session_state"],
    }])[0]
    print("Agent:", turn["message"][:300], "…")
    print("done:", turn["done"])

## Client integration

| Field | Description |
|---|---|
| `action` | `start` · `send` · `reset` |
| `session_id` | Conversation / user correlation id |
| `message` | User reply (empty for `start`) |
| `session_state` | Opaque JSON — store client-side, send back each turn |
| `output_json` | When `done=true`: JSON with `sdd` and `scorecard` markdown |

Use `mock` backend for tests. Set `LLM_BACKEND = "anthropic"` and mount `ANTHROPIC_API_KEY` on the endpoint for production.